In [1]:
import os
import requests
from dotenv import load_dotenv
from pymongo import MongoClient
import certifi

# google ADK imports
from google.adk.agents import LlmAgent
from google.adk.tools import FunctionTool
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from google.adk.models.lite_llm import LiteLlm

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), ".env"))



True

In [47]:
# MongoDB
MONGODB_URI = os.getenv("MONGODB_URI")
DB_NAME = "tokoku_db"
MANUAL_COLLECTION = "manual_chunks"
PRODUCTS_COLLECTION = "products"
MANUAL_INDEX_NAME = "manual_vector_index"
PRODUCTS_INDEX_NAME = "product_vector_index"

# Ollama
OLLAMA_URL = "http://localhost:11434/api/embeddings"
EMBEDDING_MODEL = "nomic-embed-text-v2-moe"
EMBEDDING_DIM = 768
MAX_EMBED_CHARS = 1500

# LLM
CEREBRAS_API_KEY = os.getenv("CEREBRAS_API_KEY")
ADK_MODEL = LiteLlm(model="cerebras/qwen-3-235b-a22b-instruct-2507")

print("MONGODB_URI  :", "✅" if MONGODB_URI else "❌ Not found")
print("CEREBRAS_API_KEY  :", "✅" if CEREBRAS_API_KEY else "❌ Not found")

MONGODB_URI  : ✅
CEREBRAS_API_KEY  : ✅


In [48]:
# Establish Connection to MongoDB
client = MongoClient(MONGODB_URI, tlsCAFile=certifi.where())
db = client[DB_NAME]

manual_collection   = db[MANUAL_COLLECTION]
products_collection = db[PRODUCTS_COLLECTION]

try:
    client.admin.command("ping")
    print("✅ Connected to MongoDB Atlas")
    print(f"   Database   : {DB_NAME}")
    print(f"   Collections: {MANUAL_COLLECTION}, {PRODUCTS_COLLECTION}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Connected to MongoDB Atlas
   Database   : tokoku_db
   Collections: manual_chunks, products


In [49]:
# Function for embedding user's question
def embed_query(text: str) -> list[float]:
    """Embed a query string using Ollama."""
    if len(text) > MAX_EMBED_CHARS:
        text = text[:MAX_EMBED_CHARS]

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": EMBEDDING_MODEL,
            "prompt": text
        }
    )
    response.raise_for_status()
    return response.json()["embedding"]

In [50]:
# Function for formatting the customer manual retrieval output
def format_manual_results(results: list) -> str:
    """Format manual search results into readable string for the agent."""
    if not results:
        return "No relevant information found in the Tokoku user manual."

    formatted = []
    for i, r in enumerate(results):
        formatted.append(
            f"[{i+1}] Topic: {r['topic']}\n"
            f"    Information: {r['information']}"
        )
    return "\n\n".join(formatted)

In [51]:
# Function for formatting the product retrieval output
def format_product_results(results: list) -> str:
    """Format product search results into readable string for the agent."""
    if not results:
        return "No products found matching the search criteria."

    formatted = []
    for i, r in enumerate(results):
        stock = "In Stock" if not r.get("out_of_stock") else "Out of Stock"
        formatted.append(
            f"[{i+1}] {r['title']}\n"
            f"    Brand    : {r['brand']}\n"
            f"    Category : {r['sub_category']}\n"
            f"    Price    : {r['selling_price_formatted']} "
            f"(was {r['actual_price_formatted']}, {r['discount']})\n"
            f"    Stock    : {stock}\n"
            f"    Relevance: {r.get('score', 0):.2f}\n"
            f"    {r['description'][:150]}..."
        )
    return "\n\n".join(formatted)

In [52]:
# Building the search_manual Function Tool for the Agent
def search_manual(query: str = "", **kwargs) -> str:
    """Search the Tokoku user manual to answer customer support questions.
    Use this when the customer asks about:
    - how to use a platform feature
    - account issues (login, registration, verification)
    - order tracking, cancellation, or modification
    - payment methods or billing problems
    - return and refund policies
    - seller-related questions
    - shipping and delivery questions
    - platform policies and rules

    Do NOT use this for product recommendations or catalog searches.

    Args:
        query: the customer's question in natural language
    """
    if not query:
        query = kwargs.get("query", "")
    if isinstance(query, dict):
        query = query.get("query", "")

    results = list(manual_collection.aggregate([
        {
            "$vectorSearch": {
                "index": MANUAL_INDEX_NAME,
                "path": "embedding",
                "queryVector": embed_query(query),
                "numCandidates": 50,
                "limit": 2
            }
        },
        {
            "$project": {
                "topic": 1,
                "information": 1,
                "score": {"$meta": "vectorSearchScore"}
            }
        }
    ]))
    return format_manual_results(results)

In [53]:
# Testing the search_manual Function Tool
print("Testing search_manual...")
print(search_manual("how do I return a product?"))

Testing search_manual...
[1] Topic: Return Policy
    Information: Tokoku provides buyer protection through a fair and transparent return policy. The return policy has several conditions and exceptions. Items that do not match the description, are damaged or have a manufacturing defect, are incomplete, or were shipped with the wrong item are returnable within 7 days of confirmed receipt. Non-original products can be returned if the seller agrees, but items that are not returnable include digital products and vouchers once they have been used or activated. Additionally, a money-back guarantee is provided with no time limit for non-original products.

[2] Topic: Filing a Return/Complaint
    Information: To file a return or complaint, go to My Orders and select the problematic order. Click File a Complaint and select the reason for your complaint. You will then be required to attach evidence, including up to 5 photos and 1 video up to 30 seconds, to support your claim. Next, write a clea

In [64]:
# Building the Function Tool for Hybrid Product Filtering
from typing import Optional, Literal

SubCategory = Literal[
    "Tops & T-Shirts",
    "Bottoms & Pants",
    "Footwear",
    "Jackets & Outerwear",
    "Fashion Accessories",
    "Bags & Wallets"
]

def search_products(
    query: str,
    min_price: int = 0,        
    max_price: int = 0,        
    min_discount_pct: int = 0, 
    sub_category: str = "",    
    in_stock_only: bool = False,
    limit: int = 3
) -> str:
    """Search the Tokoku product catalog to find items matching
    the customer's needs. Use this when the customer:
    - asks for product recommendations
    - mentions a category (e.g. jacket, shoes, t-shirt, bag)
    - specifies a budget or price range
    - asks about discounts or sale items
    - wants to check what is currently in stock
    - describes what they are looking for

    Do NOT use this for platform policy or account questions.

    Args:
        query          : natural language description of what the customer wants
        min_price      : minimum price in IDR integers (e.g. 100000) (0 = no minimum)
        max_price      : maximum price in IDR integers (e.g. 200000) (0 = no maximum)
        min_discount_pct: minimum discount percentage (e.g. 30 for at least 30% off) (0 = no minimum)
        sub_category   : exact category, must be one of:
                        "Tops & T-Shirts", "Bottoms & Pants", "Footwear",
                        "Jackets & Outerwear", "Fashion Accessories", "Bags & Wallets"
                        ("" = all categories)
        in_stock_only  : set True if customer explicitly wants available items only
        limit          : number of results to return, default 5, max 10.
                         Use higher value only if customer asks for more options.
    """
    if isinstance(min_price, str):
        min_price = int(min_price.replace(",", "").replace(".", ""))
    if isinstance(max_price, str):
        max_price = int(max_price.replace(",", "").replace(".", ""))
    if isinstance(min_discount_pct, str):
        min_discount_pct = int(min_discount_pct)
    if isinstance(limit, str):
        limit = int(limit)

    # debug
    # print(f"DEBUG: query={query}, max_price={max_price}, "
    #       f"sub_category={sub_category}, in_stock_only={in_stock_only}")

    # clamp limit
    limit = min(max(1, limit), 10)

    # build filter conditions
    filter_conditions = {}
    price_filter = {}
    if min_price > 0:
        price_filter["$gte"] = min_price
    if max_price > 0:
        price_filter["$lte"] = max_price
    if price_filter:
        filter_conditions["selling_price"] = price_filter
    if min_discount_pct > 0:
        filter_conditions["discount_pct"] = {"$gte": min_discount_pct}
    if sub_category:
        filter_conditions["sub_category"] = sub_category
    if in_stock_only:
        filter_conditions["out_of_stock"] = False

    # build pipeline
    vector_search_stage = {
        "$vectorSearch": {
            "index": PRODUCTS_INDEX_NAME,
            "path": "embedding",
            "queryVector": embed_query(query),
            "numCandidates": max(100, limit * 20),
            "limit": limit,
        }
    }

    # only add filter if conditions exist
    if filter_conditions:
        vector_search_stage["$vectorSearch"]["filter"] = filter_conditions

    pipeline = [
        vector_search_stage,
        {
            "$project": {
                "title": 1,
                "brand": 1,
                "sub_category": 1,
                "selling_price_formatted": 1,
                "actual_price_formatted": 1,
                "discount": 1,
                "discount_pct": 1,
                "out_of_stock": 1,
                "description": 1,
                "score": {"$meta": "vectorSearchScore"}
            }
        }
    ]

    results = list(products_collection.aggregate(pipeline))
    return format_product_results(results)

In [65]:
# Testing the product filtering Function Tool
print("Testing search_products...")
print(search_products("warm jacket", max_price=200000))

Testing search_products...
[1] Full Sleeve Striped Men Casual Jacket
    Brand    : JACK AND HAR
    Category : Jackets & Outerwear
    Price    : Rp 133.000 (was Rp 570.000, 77% off)
    Stock    : Out of Stock
    Relevance: 0.71
    This winter jacket for men is made from finest fabric, jacket is light in weight and will ensure comfort all day long. This jacket for men is smart pi...

[2] Full Sleeve Printed Men Casual Jacket
    Brand    : TOM BU
    Category : Jackets & Outerwear
    Price    : Rp 190.000 (was Rp 475.000, 60% off)
    Stock    : In Stock
    Relevance: 0.70
    Good quality jacket  for you.It gives you good comfort and a classy look...

[3] Welwear suncool1009 Nylon Arm Warmer  (Multicolor)
    Brand    : Hammer
    Category : Jackets & Outerwear
    Price    : Rp 28.000 (was Rp 57.000, 51% off)
    Stock    : In Stock
    Relevance: 0.70
    Tatto Sleeve Protect From Uv Raditaion And Also Provide Soothing Feeling In Hot Condition...


In [66]:
tokoku_agent = LlmAgent(
    name="tokoku_assistant",
    model=ADK_MODEL,
    instruction="""You are a friendly and helpful customer assistant for Tokoku,
an e-commerce platform.

Tools available:
- search_manual: for policies, account, orders, payments, shipping, returns
- search_products: for product recommendations, price/discount/stock filters
  (currently: fashion and apparel only)

Rules:
- ALWAYS call a tool first. Never answer from memory.
- For any product mention → call search_products immediately
- For any platform question → call search_manual immediately
- Respond in the same language as the customer""",
    tools=[
        FunctionTool(search_manual),
        FunctionTool(search_products)
    ]
)

In [67]:
# Build Runner and Session Service
session_service = InMemorySessionService()

runner = Runner(
    agent=tokoku_agent,
    session_service=session_service,
    app_name="tokoku_assistant"
)

In [68]:
# Create a test session
import uuid
SESSION_ID = str(uuid.uuid4())
USER_ID    = "test_user"

await session_service.create_session(
    app_name="tokoku_assistant",
    user_id=USER_ID,
    session_id=SESSION_ID
)

print(f"✅ Runner and session ready")
print(f"   Session ID : {SESSION_ID}")
print(f"   User ID    : {USER_ID}")

✅ Runner and session ready
   Session ID : 3891bb12-1632-4952-ab5f-64b198ed16e0
   User ID    : test_user


In [69]:
def chat(user_input: str) -> str:
    """Send a message to the agent and return the final response."""
    response = runner.run(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=types.Content(
            role="user",
            parts=[types.Part(text=user_input)]
        )
    )

    final_response = "No response received."
    for event in response:
        # check if this is the final response event from the agent
        if event.is_final_response():
            # extract text from the content parts
            if event.content and event.content.parts:
                final_response = event.content.parts[0].text
                break
    return final_response

In [72]:
# Testing the agent
test_queries = [
    "How do I return a product?",
    "Rekomendasikan jaket hangat untuk saya di bawah Rp 200.000"
]

import time

for query in test_queries:
    print(f"{'='*60}")
    print(f"User : {query}")
    print(f"Agent: {chat(query)}")
    print()
    time.sleep(15)  # wait 15 seconds between queries

User : How do I return a product?
Agent: Untuk mengembalikan produk di Tokoku, ikuti langkah-langkah berikut:

1. Buka **My Orders** di akun Anda.
2. Pilih pesanan dari produk yang ingin dikembalikan.
3. Klik **Ajukan Komplain**.
4. Pilih alasan komplain (misalnya: ukuran tidak pas, produk tidak sesuai deskripsi, dll).
5. Unggah bukti pendukung: hingga **5 foto** dan **1 video (maks 30 detik)**.
6. Tuliskan penjelasan yang jelas dan detail tentang masalahnya.
7. Pilih solusi yang diinginkan: **Pengembalian Uang**, **Penukaran**, atau keduanya.
8. Kirim komplain Anda.

Penjual memiliki waktu **2 x 24 jam** untuk menanggapi.  
Jika penjual tidak merespons atau Anda tidak mencapai kesepakatan, **Customer Service Tokoku** akan turun tangan untuk memediasi.

⚠️ Ingat: Produk harus dikembalikan dalam waktu **7 hari setelah penerimaan** jika ada masalah (rusak, cacat, salah kirim). Jika hanya karena ukuran tidak pas, pengembalian tergantung pada persetujuan penjual.

Butuh bantuan lebih lanju

In [71]:
# Test conversation memory
print("Testing multi-turn conversation memory...")
print()

memory_test = [
    "Show me some jackets",
    "Which one is the cheapest?",
    "Can I return it if it doesn't fit?",
]

for query in memory_test:
    print(f"User : {query}")
    print(f"Agent: {chat(query)}")
    print()

Testing multi-turn conversation memory...

User : Show me some jackets
Agent: Berikut beberapa rekomendasi jaket yang tersedia di Tokoku:

1. **Sleeveless Colorblock Men Casual Jacket**  
   - Merek: JACK AND HAR  
   - Harga: Rp 104.000 (diskon 73%)  
   - Stok: Tersedia  

2. **Sleeveless Self Design Men Nehru Jacket**  
   - Merek: shiwam ethn  
   - Harga: Rp 180.000 (diskon 41%)  
   - Stok: Tersedia  

3. **Full Sleeve Solid Men Casual Jacket**  
   - Merek: Szto  
   - Harga: Rp 570.000 (diskon 50%)  
   - Stok: Tersedia  

4. **Full Sleeve Printed Men Casual Jacket**  
   - Merek: TOM BU  
   - Harga: Rp 190.000 (diskon 60%)  
   - Stok: Tersedia  

Jaket-jaket ini memiliki gaya kasual dan tersedia dalam berbagai desain menarik. Jika Anda mencari jaket untuk cuaca dingin atau dengan material tertentu (misalnya tahan air atau insulated), beri tahu saya agar saya bisa bantu cari yang lebih sesuai!

User : Which one is the cheapest?
Agent: The cheapest jacket among the recommended